# Colab Setup

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [2]:
# --- 1. Verify and install dependencies ---
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  ✅ {package} is installed.")
    except ImportError:
        print(f"  ❌ {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  ✅ {package} is now installed.")
        except Exception as e:
            print(f"  ❌ Failed to install {package}: {e}")

# Special check for PyTorch CUDA
print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  ❌ PyTorch is not installed.")

print("Verification complete.")

Verifying and installing missing packages if necessary...
  ✅ numpy is installed.
  ✅ matplotlib is installed.
  ✅ librosa is installed.
  ✅ tqdm is installed.
  ✅ sklearn is installed.
  ✅ stempeg is installed.
  ✅ torch is installed.
  ✅ torchvision is installed.
  ✅ torchaudio is installed.
  ✅ musdb is installed.

--- PyTorch CUDA status ---
  ⚠️ PyTorch is installed, but CUDA is NOT available.
Verification complete.


## 1. Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [14]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import importlib

# Simple, fixed project layout (data/, checkpoints/ at repo root)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "local_main.ipynb").exists():
    for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if (p / "local_main.ipynb").exists():
            PROJECT_ROOT = p
            break
os.chdir(PROJECT_ROOT)

DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import models.utils as utils
importlib.reload(utils)
from models import model_A as ma

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Checkpoints dir: {CHECKPOINT_DIR.resolve()}")

Using device: cpu
Project root: c:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning
Data dir: C:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning\data
Checkpoints dir: C:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning\checkpoints


## 2. Data Loading and Preprocessing

**Two modes of operation:**

1. **Training Mode** (`DOWNLOAD_DATA = True`): 
   - Downloads MUSDB18 dataset (~4GB, 144 tracks)
   - Prepares training data with realistic mixture weights
   - Required for training from scratch
   
2. **Inference-Only Mode** (`DOWNLOAD_DATA = False`):
   - Skip data download (default)
   - Use existing model checkpoints for demonstrations
   - Upload your own songs for separation
   - Perfect for quick demos and evaluation

**Mixture Weights** (for those who download):
- Vocals: 35%, Drums: 30%, Bass: 20%, Other: 15%
- Matches real music production standards

In [ ]:
DOWNLOAD_DATA = True  # Change to True to download MUSDB18 dataset (~4GB)
FORCE_REBUILD = False  # Set True to regenerate data even if it exists

if DOWNLOAD_DATA:
    import musdb
    print("Downloading MUSDB18 dataset (~4GB)...")
    mus = musdb.DB(download=True)
    utils.prepare_curriculum_cache(mus=mus, cache_dir=str(DATA_DIR), sr=22050, force_rebuild=FORCE_REBUILD)
    print(f"✅ Dataset ready: {len(mus.tracks)} tracks")

# Load file lists if data exists
try:
    mix_files_stage1, tgt_files_stage1, mix_files_stage2, tgt_files_stage2 = utils.get_curriculum_file_lists(cache_dir=str(DATA_DIR))
    print(f"✅ Training data found: Stage 1 ({len(mix_files_stage1)}), Stage 2 ({len(mix_files_stage2)})")
except Exception as e:
    print("⚠️  No training data found (OK for inference-only mode)")
    print(f"   Details: {e}")
    mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

Generating Curriculum Cache at data...


## 3. Overfit Sanity Check

Test model's ability to overfit on a single song (validates implementation).

In [5]:
# Get overfit configuration from utils
OVERFIT_CONFIG = utils.get_overfit_config()

overfit_device = device
overfit_processor = utils.AudioProcessor(device=overfit_device)

overfit_model = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=128,
    num_layers=4,
    batchnorm=True,
    dropout=0.0
).to(overfit_device)

overfit_loss_fn = nn.MSELoss()
overfit_optimizer = optim.Adam(overfit_model.parameters(), lr=OVERFIT_CONFIG['learning_rate'])

print("Overfit setup complete.")

Overfit setup complete.


### Run Overfit Training

Only runs if checkpoint doesn't exist. Set `SKIP_OVERFIT = True` to skip.

In [6]:
SKIP_OVERFIT = False  # Set to True to skip overfit test

if not SKIP_OVERFIT:
    # Check if checkpoint exists
    overfit_ckpt = CHECKPOINT_DIR / "debug_overfit_1song.pth"
    print(f"\n{'='*60}")
    print("Overfit Sanity Check")
    print(f"{'='*60}")
    checkpoint_exists = utils.check_checkpoint(overfit_ckpt, "Overfit Checkpoint")
    
    if not checkpoint_exists and len(mix_files_stage1) > 0:
        print("\nStarting Overfit Test on 1 Random Song...")
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(DATA_DIR),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )
    elif len(mix_files_stage1) == 0:
        print("\n⚠️  No training data available. Skipping overfit test.")
        print("   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.")
        history_overfit = {}
    else:
        history_overfit = {}
else:
    print("Skipping overfit test (SKIP_OVERFIT = True)")
    history_overfit = {}


Overfit Sanity Check
⏳ Overfit Checkpoint not found: checkpoints\debug_overfit_1song.pth
   Training will start...

⚠️  No training data available. Skipping overfit test.
   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.


### View Overfit Results

In [7]:
# Plot overfit learning curve (if available)
utils.plot_loss_history(history_overfit, title="Overfit Learning Curve (1 Song)")

No training data found for Overfit Learning Curve (1 Song)


## 4. Model Architecture

View the U-Net model structure and parameter count.

In [8]:
# Model A architecture summary
model_summary = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=64,
    num_layers=4
).to(device)
print(model_summary)
del model_summary

TimeFrequencyDomainUNet(
  (encoders): ModuleList(
    (0): EncoderBlock(
      (block): Sequential(
        (0): ConvLayer2D(
          (block): Sequential(
            (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU()
          )
        )
        (1): ConvLayer2D(
          (block): Sequential(
            (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU()
          )
        )
      )
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): EncoderBlock(
      (block): Sequential(
        (0): ConvLayer2D(
          (block): Sequential(
            (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): BatchNorm2d(128, eps=1e

## 5. Model Configuration

Configure model architecture and training hyperparameters.

In [9]:
# Get configurations from utils
MODEL_CONFIG = utils.get_model_a_config()
TRAIN_CONFIG = utils.get_training_config()

processor = utils.AudioProcessor(device=device)

model = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=64,
    num_layers=4,
    batchnorm=True,
    dropout=0.1
).to(device)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=TRAIN_CONFIG['learning_rate'])

print("Model A setup complete.")

Model A setup complete.


## 6. Full Training (Curriculum Learning)

Train the model on the full MUSDB18 dataset in two stages.

In [10]:
SKIP_TRAINING = False  # Set to True to skip full training

if not SKIP_TRAINING:
    # Check if checkpoints exist
    ckpt_s1 = CHECKPOINT_DIR / "full_stage1.pth"
    ckpt_s2 = CHECKPOINT_DIR / "full_stage2.pth"
    print(f"\n{'='*60}")
    print("Full Training: Curriculum Learning")
    print(f"{'='*60}")
    
    s1_exists = utils.check_checkpoint(ckpt_s1, "Stage 1 Checkpoint")
    s2_exists = utils.check_checkpoint(ckpt_s2, "Stage 2 Checkpoint")
    
    # Only train if data exists and checkpoints don't
    if len(mix_files_stage1) > 0 and len(mix_files_stage2) > 0:
        if not s1_exists or not s2_exists:
            print("\nStarting Full Training Pipeline...")
            hist_s1, hist_s2 = utils.run_full_training(
                model=model,
                processor=processor,
                optimizer=optimizer,
                loss_fn=loss_fn,
                train_config=TRAIN_CONFIG,
                cache_dir=str(DATA_DIR),
                save_path_stage1=str(ckpt_s1),
                save_path_stage2=str(ckpt_s2),
                device=device,
            )
        else:
            print("\n✅ Both checkpoints exist. Training skipped.")
            hist_s1, hist_s2 = {}, {}
    else:
        print("\n⚠️  No training data available. Skipping full training.")
        print("   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.")
        hist_s1, hist_s2 = {}, {}
else:
    print("Skipping full training (SKIP_TRAINING = True)")
    hist_s1, hist_s2 = {}, {}


Full Training: Curriculum Learning
⏳ Stage 1 Checkpoint not found: checkpoints\full_stage1.pth
   Training will start...
⏳ Stage 2 Checkpoint not found: checkpoints\full_stage2.pth
   Training will start...

⚠️  No training data available. Skipping full training.
   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.


## 7. View Training Results

Plot loss curves from saved checkpoint.

In [11]:
utils.plot_loss_from_checkpoint(str(CHECKPOINT_DIR / "full_stage2.pth"))

c:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning\models\utils.py:771: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt =

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/full_stage2.pth'

## 8. Listen to Separated Audio

Demo the model on MUSDB18 samples with audio playback and spectrograms.

In [ ]:
# Preview a sample separation from the cache
utils.demo_separation_sample(
    model=model,
    processor=processor,
    cache_dir=str(DATA_DIR),
    stage="stage1",
    song_num=12,
    duration=6,
    sr=22050,
    device=device,
    play_audio_output=True,
 )

In [ ]:
import os, sys
print('Current working directory:', os.getcwd())
print('sys.path:', sys.path)
print('Directory listing:', os.listdir('.'))

# Model B2: [Architecture Name]

[Description of Model B2]

# Model B1: [Architecture Name]

[Description of Model B1]